# 03 — Zero-shot baseline: Qwen2.5-1.5B-Instruct

Prompts an instruction-tuned LLM with the full inventory of 60 intents and asks it to name
one. No training, no examples in the prompt — this measures what the model already knows
about Turkish voice commands, and sets the bar that LoRA fine-tuning in notebook 04 has to
clear.

The prompt lives in `src/prompting.py` and is shared with notebook 04, so the only thing
that differs between zero-shot and fine-tuned is the weights.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alper4n/turkish-intent-llm/blob/main/notebooks/03_zeroshot_qwen.ipynb)

## 0. Setup

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/alper4n/turkish-intent-llm.git"
    if not Path("turkish-intent-llm").exists():
        !git clone -q $REPO_URL
    %cd turkish-intent-llm
    !pip install -q -U "transformers>=5.0,<6" "datasets>=3.0" "accelerate>=1.0" "pyyaml>=6.0"

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root:", root)

/content/turkish-intent-llm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.7 MB/s eta 0:00:00
repo root: /content/turkish-intent-llm


In [2]:
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data import load_splits, intent_labels
from src.metrics import compute_metrics
from src.prompting import (build_messages, build_system_prompt, generate_labels,
                           parse_label, prepare_tokenizer)
from src.utils import describe_environment, load_config, save_results, set_seed

data_cfg = load_config(root / "configs" / "data.yaml")
cfg = load_config(root / "configs" / "zeroshot_qwen.yaml")
SEED = data_cfg["seed"]
set_seed(SEED)

FIG_DIR = root / data_cfg["paths"]["figures_dir"]
FIG_DIR.mkdir(parents=True, exist_ok=True)

env = describe_environment()
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
env["device"] = DEVICE

# Measured for 8 GB unified memory; a 16 GB T4 has room for a much larger batch.
BATCH = (cfg["generation"]["batch_size_cuda"] if DEVICE == "cuda"
         else cfg["generation"]["batch_size"])
env["batch_size"] = BATCH
print(json.dumps(env, indent=2))

{
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "bf16_supported": true,
  "device": "cuda",
  "batch_size": 16
}


## 1. Data and label inventory

In [3]:
splits = load_splits(data_cfg["dataset"]["locale"],
                     cache_dir=root / data_cfg["dataset"]["cache_dir"])
LABELS = intent_labels(splits["train"])
print(f"{len(LABELS)} intents, {len(splits['test']):,} test utterances")

60 intents, 2,974 test utterances


## 2. The prompt

One system message carrying all 60 intent names, one user message carrying the utterance.
Decoding is greedy, so the run is reproducible.

In [4]:
system_prompt = build_system_prompt(LABELS)
print(system_prompt[:400] + "\n...\n")
print("--- example user turn ---")
print(build_messages("yarın sabah yedide alarm kur", LABELS)[1]["content"])

You are an intent classifier for a Turkish voice assistant.
Classify the user's utterance into exactly one of these intents:
alarm_query, alarm_remove, alarm_set, audio_volume_down, audio_volume_mute, audio_volume_other, audio_volume_up, calendar_query, calendar_remove, calendar_set, cooking_query, cooking_recipe, datetime_convert, datetime_query, email_addcontact, email_query, email_querycontact,
...

--- example user turn ---
Utterance: "yarın sabah yedide alarm kur"


## 3. Load the model

fp16 on every backend. The T4 is Turing (compute capability 7.5), where bf16 runs only
under emulation — native bf16 starts at Ampere. fp16 *inference* is also stable on
Apple MPS, unlike fp16 training, which is why notebook 02 stays in fp32 without CUDA.

In [5]:
MODEL_NAME = cfg["model"]["name"]

tokenizer = prepare_tokenizer(AutoTokenizer.from_pretrained(MODEL_NAME))
start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=getattr(torch, cfg["model"]["dtype"]))
model.to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded in {time.time()-start:.0f}s")
print(f"parameters: {n_params:,}  (trainable in this experiment: 0)")
print(f"prompt length: {len(tokenizer(build_messages('test', LABELS)[0]['content'])['input_ids'])} tokens")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded in 41s
parameters: 1,543,714,304  (trainable in this experiment: 0)
prompt length: 270 tokens


## 4. Check the prompt on validation, not on test

The prompt format is verified on a validation subsample first. Iterating on the test set
would quietly turn this "zero-shot" baseline into a tuning run with an unreported search,
and its number would no longer be comparable with the other two experiments.

In [6]:
dev_n = cfg["prompt"]["dev_sample_size"]
dev = splits["validation"].sample(dev_n, random_state=SEED).reset_index(drop=True)

dev_raw = generate_labels(
    model, tokenizer, dev["utt"].tolist(), LABELS,
    batch_size=BATCH,
    max_new_tokens=cfg["generation"]["max_new_tokens"],
    device=DEVICE,
)
dev_pred = [parse_label(text, LABELS) for text in dev_raw]

dev_valid = sum(p in set(LABELS) for p in dev_pred)
dev_acc = sum(p == g for p, g in zip(dev_pred, dev["intent"])) / len(dev)
print(f"\nvalid label outputs: {dev_valid}/{len(dev)} ({dev_valid/len(dev):.1%})")
print(f"validation accuracy (n={len(dev)}): {dev_acc:.4f}")

  done: 200 utterances in 0.2 min (13 batches)

valid label outputs: 193/200 (96.5%)
validation accuracy (n=200): 0.4550


In [7]:
print("raw outputs that did not parse to a known intent:")
unparsed = [(raw, parsed) for raw, parsed in zip(dev_raw, dev_pred)
            if parsed not in set(LABELS)]
if unparsed:
    for raw, parsed in unparsed[:10]:
        print(f"  {raw!r} -> {parsed!r}")
else:
    print("  (none — the model always answered with a valid intent name)")

raw outputs that did not parse to a known intent:
  'audio_volume_off' -> 'audio_volume_off'
  'qa_game' -> 'qa_game'
  'transport_radio' -> 'transport_radio'
  'audio_volume_off' -> 'audio_volume_off'
  'quirky' -> 'quirky'
  'audio_volume_dim' -> 'audio_volume_dim'
  'qa_math' -> 'qa_math'


The prompt is now frozen. Everything below runs once on the test set.

## 5. Test set

In [8]:
test_raw = generate_labels(
    model, tokenizer, splits["test"]["utt"].tolist(), LABELS,
    batch_size=BATCH,
    max_new_tokens=cfg["generation"]["max_new_tokens"],
    device=DEVICE,
)
y_pred = [parse_label(text, LABELS) for text in test_raw]
y_true = splits["test"]["intent"].tolist()

  320/2974 utterances | 0.3 min elapsed | ~2.7 min left
  640/2974 utterances | 0.7 min elapsed | ~2.4 min left
  960/2974 utterances | 1.0 min elapsed | ~2.2 min left
  1280/2974 utterances | 1.4 min elapsed | ~1.9 min left
  1600/2974 utterances | 1.8 min elapsed | ~1.5 min left
  1920/2974 utterances | 2.1 min elapsed | ~1.2 min left
  2240/2974 utterances | 2.5 min elapsed | ~0.8 min left
  2560/2974 utterances | 2.9 min elapsed | ~0.5 min left
  2880/2974 utterances | 3.2 min elapsed | ~0.1 min left
  done: 2974 utterances in 3.3 min (186 batches)


In [9]:
scored = compute_metrics(y_true, y_pred, LABELS)

print(f"test utterances      : {scored['n_examples']:,}")
print(f"accuracy             : {scored['accuracy']:.4f}")
print(f"macro-F1 (59 present): {scored['macro_f1']:.4f}")
print(f"macro-F1 (all 60)    : {scored['macro_f1_all_labels']:.4f}")
print(f"invalid predictions  : {scored['label_space']['n_invalid_predictions']}")
for item in scored["label_space"]["invalid_predictions"][:5]:
    print(f"    {item['count']:>4}  {item['prediction']!r}")

test utterances      : 2,974
accuracy             : 0.4526
macro-F1 (59 present): 0.4049
macro-F1 (all 60)    : 0.3982
invalid predictions  : 101
       6  'transport_radio'
       5  'transport_order'
       5  'maths'
       4  'audio_volume_off'
       4  'iot_hue_lightclean'


### Predictions are saved

A generative model's raw output is data in its own right — the error-analysis notebook needs
the actual strings, not just the scores.

In [10]:
predictions = pd.DataFrame({
    "id": splits["test"]["id"],
    "utt": splits["test"]["utt"],
    "gold": y_true,
    "raw_output": [text.strip() for text in test_raw],
    "predicted": y_pred,
})
predictions["correct"] = predictions["gold"] == predictions["predicted"]

pred_path = (root / data_cfg["paths"]["results_dir"]
             / f"{cfg['output']['predictions_name']}.csv")
predictions.to_csv(pred_path, index=False)
print("wrote", pred_path)
predictions.head(8)

wrote /content/turkish-intent-llm/results/zeroshot_qwen_predictions.csv


,id,utt,gold,raw_output,predicted,correct
0,0,bu hafta beni sabah beşte uyandır,alarm_set,alarm_set,alarm_set,True
1,3,sessiz,audio_volume_mute,general_greet,general_greet,False
2,8,tek ihtiyacımız olan pembe,iot_hue_lightchange,recommendation_movies,recommendation_movies,False
3,14,ve karanlık çöktü,iot_hue_lighton,alarm_set,alarm_set,False
4,19,olly yatak odasındaki ışıkları kapat,iot_hue_lightoff,iot_hue_lightoff,iot_hue_lightoff,True
5,27,burası çok pis biraz gürültü yap,iot_cleaning,audio_volume_up,audio_volume_up,False
6,31,koridor u süpür,iot_cleaning,iot_hue_lightoff,iot_hue_lightoff,False
7,41,ekran parlaklığımın statüsünü istiyorum,general_quirky,audio_volume_other,audio_volume_other,False


## 6. Where zero-shot fails

In [11]:
per_class = pd.DataFrame(scored["per_class"])
print("Worst 12 intents by F1:")
print(per_class.head(12).to_string(index=False))
print()
print(f"intents with F1 = 0: {(per_class['f1'] == 0).sum()} of {len(per_class)}")

Worst 12 intents by F1:
            intent  precision  recall     f1  support
      iot_wemo_off     0.0000  0.0000 0.0000       18
       iot_wemo_on     0.0000  0.0000 0.0000       10
     general_greet     0.0000  0.0000 0.0000        1
    general_quirky     0.2308  0.0178 0.0330      169
      iot_cleaning     0.2000  0.0385 0.0645       26
      general_joke     0.0536  0.1579 0.0800       19
   iot_hue_lighton     0.0476  0.6667 0.0889        3
    music_settings     0.0769  0.1667 0.1053        6
 music_dislikeness     0.0667  0.2500 0.1053        4
email_querycontact     0.1026  0.1538 0.1231       26
        qa_factoid     0.3929  0.0780 0.1302      141
   iot_hue_lightup     0.6667  0.0741 0.1333       27

intents with F1 = 0: 3 of 59


In [12]:
print("Most frequent confusions (gold -> predicted):")
for row in scored["top_confusions"]:
    print(f"  {row['count']:>3}  {row['gold']}  ->  {row['predicted']}")

Most frequent confusions (gold -> predicted):
   79  play_music  ->  music_query
   45  calendar_query  ->  calendar_set
   30  qa_factoid  ->  weather_query
   27  general_quirky  ->  general_joke
   27  email_query  ->  email_querycontact
   24  play_music  ->  play_podcasts
   23  calendar_set  ->  recommendation_events
   23  email_sendemail  ->  email_addcontact
   21  calendar_set  ->  alarm_set
   18  calendar_remove  ->  recommendation_events
   18  calendar_query  ->  recommendation_events
   16  qa_factoid  ->  news_query
   15  calendar_set  ->  calendar_query
   15  cooking_recipe  ->  cooking_query
   15  qa_factoid  ->  calendar_query


In [13]:
# A generative classifier can collapse onto a few "attractor" labels regardless of input.
pred_counts = predictions["predicted"].value_counts()
gold_counts = predictions["gold"].value_counts()
comparison = pd.DataFrame({"predicted": pred_counts, "gold": gold_counts}).fillna(0).astype(int)
comparison["over_prediction"] = comparison["predicted"] - comparison["gold"]
print("Most over-predicted intents (model says it far more often than it occurs):")
print(comparison.nlargest(10, "over_prediction").to_string())
print()
print(f"distinct intents ever predicted: {predictions['predicted'].nunique()} of {len(LABELS)}")

Most over-predicted intents (model says it far more often than it occurs):
                       predicted  gold  over_prediction
music_query                  140    35              105
recommendation_events        100    43               57
alarm_set                     96    41               55
cooking_query                 53     0               53
news_query                   172   124               48
recommendation_movies         64    20               44
transport_query               95    51               44
email_addcontact              52    12               40
iot_hue_lighton               42     3               39
general_joke                  56    19               37

distinct intents ever predicted: 123 of 60


## 7. Against the fine-tuned baseline

In [14]:
berturk_path = root / data_cfg["paths"]["results_dir"] / "berturk.json"
rows = [{
    "experiment": "Zero-shot Qwen2.5-1.5B",
    "trainable_params": 0,
    "accuracy": scored["accuracy"],
    "macro_f1": scored["macro_f1"],
}]
if berturk_path.exists():
    berturk = json.loads(berturk_path.read_text(encoding="utf-8"))
    rows.append({
        "experiment": "BERTurk fine-tune",
        "trainable_params": berturk["training"]["trainable_parameters"],
        "accuracy": berturk["test"]["accuracy"],
        "macro_f1": berturk["test"]["macro_f1"],
    })
pd.DataFrame(rows).set_index("experiment")

,trainable_params,accuracy,macro_f1
experiment,,,
Zero-shot Qwen2.5-1.5B,0,0.4526,0.4049
BERTurk fine-tune,110663484,0.8773,0.8501


## 8. Save results

In [15]:
payload = {
    "experiment": cfg["experiment"],
    "model": MODEL_NAME,
    "seed": SEED,
    "config": cfg,
    "environment": env,
    "model_info": {
        "total_parameters": int(n_params),
        "trainable_parameters": 0,
        "dtype": cfg["model"]["dtype"],
    },
    "prompt": {
        "system": system_prompt,
        "dev_sample_size": dev_n,
        "dev_accuracy": round(float(dev_acc), 4),
        "dev_valid_rate": round(dev_valid / len(dev), 4),
    },
    "test": scored,
    "prediction_diversity": {
        "distinct_intents_predicted": int(predictions["predicted"].nunique()),
        "n_intents": len(LABELS),
    },
}

path = save_results(payload, cfg["output"]["results_name"],
                    results_dir=root / data_cfg["paths"]["results_dir"])
print("wrote", path)
print(f"\nHEADLINE  accuracy={scored['accuracy']:.4f}  macro-F1={scored['macro_f1']:.4f}")

wrote /content/turkish-intent-llm/results/zeroshot_qwen.json

HEADLINE  accuracy=0.4526  macro-F1=0.4049


### Downloading the results

Colab sessions are ephemeral, so retrieve the results before this one ends. Running the
cell below starts both downloads; to keep the executed notebook itself, use
**File → Download → Download .ipynb**.

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(path))
    files.download(str(pred_path))
else:
    print("Not on Colab — results are already on disk:")
    print(" ", path)
    print(" ", pred_path)